# 🔬 Pattern Hunters: How the Michaelis-Menten Equation Emerges from Nature
### A Self-Guided Notebook for BSc / MSc Biochemistry Students
**Kuchinda Degree College | Department of Zoology**

---

> *"Mathematics does not describe nature from the outside. It is forced out of nature by the act of careful questioning."*

---

## How to use this notebook

This is **not** a notebook that gives you equations to memorise.  
It is a notebook that asks you questions — and shows you that the equation was the **only honest answer**.

Each section begins with an **observation** or a **question**.  
Run each cell. Read the output. Think before moving to the next cell.

**The single figure in your textbook** (v vs [S], the hyperbolic curve) is the *last* thing we will arrive at — not the first. By the time you see it, you will understand exactly why it has to look the way it does.

---

### Contents
1. The Real Experiment — What Actually Happens in One Flask
2. The Problem with Measuring Late
3. Five Things That Go Wrong Simultaneously
4. The Architecture of the MM Experiment — Many Flasks, Not One
5. The Data Demands a Shape
6. The Mechanism Explains the Shape
7. The Equation is Cornered — It Cannot Be Anything Else
8. The Equation Becomes a Tool — Reading Biology from Parameters
9. 🧪 Your Own Experiment — Change the Parameters, Hunt the Pattern
10. 🌍 Western Odisha Connection — Soil Enzymes as Biomarkers

In [ ]:
# ============================================================
# RUN THIS FIRST — installs and imports everything needed
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Global style — clean and readable
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'lines.linewidth': 2.2,
})

# Core simulation functions used throughout
def mm_rate(S, Vmax, Km):
    """Michaelis-Menten rate equation."""
    return (Vmax * S) / (Km + S)

def simulate_progress_curve(S0, Vmax, Km, t_max=60, n_points=500,
                              product_inhibition=False, Ki_p=None,
                              enzyme_decay=False, kd=0.0,
                              reverse_reaction=False, Keq=None):
    """
    Simulates a real progress curve — product formation over time.
    Returns arrays: time, product, substrate, instantaneous_rate
    """
    dt = t_max / n_points
    times = np.linspace(0, t_max, n_points + 1)
    S = S0
    P = 0.0
    S_arr, P_arr, v_arr = [S0], [0.0], [mm_rate(S0, Vmax, Km)]

    for i in range(n_points):
        t = times[i]
        # Effective Vmax considering enzyme decay
        Veff = Vmax * np.exp(-kd * t) if enzyme_decay else Vmax
        # Effective Km considering product inhibition
        Keff = Km * (1 + P / Ki_p) if product_inhibition and Ki_p else Km
        # Forward rate
        v_fwd = (Veff * S) / (Keff + S) if S > 0 else 0
        # Reverse rate
        v_rev = 0
        if reverse_reaction and Keq:
            Vrev = Vmax / Keq
            v_rev = (Vrev * P) / (Km + P) if P > 0 else 0
        v_net = max(0, v_fwd - v_rev)
        dP = v_net * dt
        P = min(P + dP, S0)
        S = max(0, S0 - P)
        S_arr.append(S)
        P_arr.append(P)
        v_arr.append(v_net)

    return times, np.array(S_arr), np.array(P_arr), np.array(v_arr)

print("✓ Setup complete. All functions loaded.")
print("\nYou are a Pattern Hunter. Your job is not to memorise equations.")
print("Your job is to notice what the data is trying to tell you.")

---
## Section 1 — The Real Experiment
### What actually happens when you mix enzyme and substrate in one flask?

Before any equation. Before any graph. Imagine this:

You take a flask. You add a fixed amount of enzyme. You add substrate.  
You measure how much **product** has formed — at 1 minute, 5 minutes, 10 minutes, 30 minutes...

**Question before you run the cell:**  
*What do you expect the graph of [Product] vs Time to look like? Draw it in your notebook first.*

Now run the cell and compare.

In [ ]:
# === SECTION 1: The Progress Curve ===
# One flask. Fixed enzyme. Watch product form over time.

Vmax = 80   # µmol/min — maximum possible rate
Km   = 25   # mM       — you will understand this soon
S0   = 120  # mM       — starting substrate concentration

times, S_arr, P_arr, v_arr = simulate_progress_curve(S0, Vmax, Km, t_max=60)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Section 1 — The Real Experiment: One Flask Over Time',
             fontsize=14, fontweight='bold', y=1.01)

# Left: Product formation
ax1 = axes[0]
ax1.plot(times, P_arr, color='#1D9E75', label='[Product] formed')
ax1.fill_between(times, P_arr, alpha=0.08, color='#1D9E75')
ax1.axhline(S0, color='#BA7517', linestyle='--', alpha=0.5, label=f'[S]₀ = {S0} mM (ceiling)')

# Highlight initial linear window
t_window = 2.0
v0_true = mm_rate(S0, Vmax, Km)
t_init = np.linspace(0, t_window, 50)
ax1.fill_between(t_init, v0_true * t_init, alpha=0.25, color='#BA7517',
                  label=f'Initial linear zone (v₀ ≈ {v0_true:.1f} µmol/min)')
ax1.plot(t_init, v0_true * t_init, color='#BA7517', linewidth=2.5)

ax1.set_xlabel('Time (min)')
ax1.set_ylabel('[Product]  (mM)')
ax1.set_title('Product formation over time')
ax1.legend(fontsize=10)
ax1.annotate('Initial rate\n(steep, linear)', xy=(1, v0_true*1),
              xytext=(8, 20), fontsize=9, color='#BA7517',
              arrowprops=dict(arrowstyle='->', color='#BA7517', lw=1.2))
ax1.annotate('Rate slows\n(substrate depleting)', xy=(20, P_arr[166]),
              xytext=(30, 50), fontsize=9, color='#555',
              arrowprops=dict(arrowstyle='->', color='#888', lw=1.2))

# Right: Instantaneous rate over time
ax2 = axes[1]
ax2.plot(times, v_arr, color='#7F77DD', label='Instantaneous rate v(t)')
ax2.axhline(v0_true, color='#1D9E75', linestyle='--', alpha=0.6,
            label=f'v₀ = {v0_true:.1f} µmol/min (true initial rate)')
ax2.fill_between([0, t_window], [0, 0], [v0_true, v0_true],
                  alpha=0.15, color='#BA7517', label='v₀ measurement window')
ax2.set_xlabel('Time (min)')
ax2.set_ylabel('Rate v  (µmol/min)')
ax2.set_title('How the rate changes as the reaction proceeds')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('section1_progress_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔍 OBSERVE and ANSWER in your notebook:")
print("  1. At what point does product formation slow down noticeably?")
print("  2. Does the rate ever reach zero? Why or why not?")
print("  3. The orange zone is the ONLY region where simple kinetics are valid.")
print("     Why does everything after it become unreliable?")
print(f"\n  True initial velocity v₀ = {v0_true:.1f} µmol/min")
print(f"  Rate at t=10 min        = {v_arr[83]:.1f} µmol/min  ← already {100*(1-v_arr[83]/v0_true):.0f}% lower")
print(f"  Rate at t=30 min        = {v_arr[250]:.1f} µmol/min  ← {100*(1-v_arr[250]/v0_true):.0f}% lower than true v₀")

---
## Section 2 — The Problem With Measuring Late

What happens if you measure the rate at t = 10 min instead of t = 0?  
You get a number — but it is **not the number the MM equation asks for**.

Five things have changed simultaneously since t = 0.  
**Select which factor to isolate. Watch what it does to the progress curve.**

In [ ]:
# === SECTION 2: Five Confounding Factors ===
# Each panel shows ONE factor operating in isolation.
# In reality, all five act simultaneously.

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Section 2 — Five Reasons Why Late Measurements Are Unreliable\n'
             '(dashed = ideal MM behaviour, solid = observed with each confound)',
             fontsize=13, fontweight='bold')

# Ideal reference
t_ref, _, P_ref, _ = simulate_progress_curve(S0, Vmax, Km, t_max=60)

configs = [
    dict(title='1. Substrate depletion\n(always present)',
         color='#E24B4A',
         kwargs=dict(),
         note='[S] falls → rate falls.\nMM assumes [S] is fixed.'),
    dict(title='2. Product inhibition\n(very common)',
         color='#BA7517',
         kwargs=dict(product_inhibition=True, Ki_p=35),
         note='Product competes with substrate.\nMM assumes [P] = 0.'),
    dict(title='3. Reverse reaction\n(thermodynamic)',
         color='#7F77DD',
         kwargs=dict(reverse_reaction=True, Keq=6),
         note='P → S accelerates as [P] builds.\nMM describes only forward rate.'),
    dict(title='4. Enzyme denaturation\n(instability over time)',
         color='#D85A30',
         kwargs=dict(enzyme_decay=True, kd=0.03),
         note='Enzyme loses activity over time.\nMM assumes [E] is constant.'),
    dict(title='5. Steady-state violated\n(mathematical foundation)',
         color='#534AB7',
         kwargs=dict(),  # shown as annotation
         note='[ES] ≈ constant only when [S] >> Km.\nAs [S] falls toward Km, MM breaks down.'),
]

flat_axes = axes.flatten()

for i, cfg in enumerate(configs):
    ax = flat_axes[i]
    # Reference (ideal)
    ax.plot(t_ref, P_ref, color='#aaaaaa', linestyle='--',
            linewidth=1.5, label='Ideal (substrate depletion only)', alpha=0.7)

    if i == 4:  # steady-state — show [S] vs Km
        _, S_ss, P_ss, _ = simulate_progress_curve(S0, Vmax, Km)
        ax.plot(t_ref, P_ss, color=cfg['color'], label='Observed')
        ax.axhline(S0 - Km, color='#1D9E75', linestyle=':', alpha=0.7,
                   label=f'[S] = Km threshold')
        ax.fill_between(t_ref, S0 - Km, S0,
                         where=P_ss <= (S0 - Km),
                         alpha=0.1, color='#1D9E75', label='Steady-state valid')
        ax.fill_between(t_ref, 0, P_ss,
                         where=P_ss > (S0 - Km),
                         alpha=0.1, color=cfg['color'], label='Steady-state fails')
    else:
        _, _, P_conf, _ = simulate_progress_curve(S0, Vmax, Km, **cfg['kwargs'])
        ax.plot(t_ref, P_conf, color=cfg['color'], label='Observed (with confound)')
        ax.fill_between(t_ref, P_ref, P_conf,
                         alpha=0.12, color=cfg['color'],
                         label='Error introduced')

    # v0 window
    ax.axvspan(0, 2, alpha=0.12, color='#BA7517', label='v₀ safe zone')
    ax.set_title(cfg['title'], fontsize=11, fontweight='bold', color=cfg['color'])
    ax.set_xlabel('Time (min)', fontsize=10)
    ax.set_ylabel('[P] (mM)', fontsize=10)
    ax.text(0.97, 0.08, cfg['note'], transform=ax.transAxes,
            fontsize=8.5, color=cfg['color'], ha='right', va='bottom',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                       edgecolor=cfg['color'], alpha=0.8))
    ax.legend(fontsize=7.5, loc='upper left')

# Summary panel
ax6 = flat_axes[5]
all_colors = ['#E24B4A','#BA7517','#7F77DD','#D85A30','#534AB7']
all_labels = ['1. Substrate depletion','2. Product inhibition',
               '3. Reverse reaction','4. Enzyme decay','5. Steady-state violated']
all_kws = [
    dict(),
    dict(product_inhibition=True, Ki_p=35),
    dict(reverse_reaction=True, Keq=6),
    dict(enzyme_decay=True, kd=0.03),
    dict(),
]
ax6.plot(t_ref, P_ref, color='#aaaaaa', linestyle='--',
         linewidth=1.5, label='Ideal', alpha=0.8)
for col, lab, kw in zip(all_colors, all_labels, all_kws):
    _, _, P_c, _ = simulate_progress_curve(S0, Vmax, Km, **kw)
    ax6.plot(t_ref, P_c, color=col, alpha=0.7, linewidth=1.5, label=lab)
ax6.axvspan(0, 2, alpha=0.15, color='#BA7517')
ax6.set_title('All five confounds together\n(the real experiment)', fontsize=11, fontweight='bold')
ax6.set_xlabel('Time (min)', fontsize=10)
ax6.set_ylabel('[P] (mM)', fontsize=10)
ax6.legend(fontsize=7.5)
ax6.text(2.5, 5, 'Only the\norange zone\nis safe', fontsize=9,
          color='#BA7517', fontweight='bold')

plt.tight_layout()
plt.savefig('section2_five_confounds.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔍 CRITICAL THINKING — answer in your notebook:")
print("  1. All five confounds act SIMULTANEOUSLY in a real experiment.")
print("     Which one do you think causes the greatest error at t = 30 min?")
print("  2. The orange zone (t = 0 to 2 min) is where ALL five are negligible.")
print("     What assumption must you make about enzyme stability during this window?")
print("  3. The textbook shows only one curve (v vs [S]).")
print("     Which of these five realities does it hide completely?")

---
## Section 3 — The Architecture of the MM Experiment
### Many flasks, not one. Each gives exactly one point.

You now understand *why* only v₀ is trustworthy.  
The next question is architectural: **how is the MM curve actually built?**

Most students — and many textbooks — imply you could do this in one flask:

> *Start with low [S], measure rate. Add more substrate. Measure again. Repeat.*

**This is wrong. Think carefully: what has changed inside the flask by the time you add more substrate?**

- Product has accumulated → product inhibition is now present
- Some substrate has been consumed → your [S] is not what you think
- The enzyme has been running for minutes → activity may have drifted
- Equilibrium has shifted → reverse reaction is now non-zero

**Every single one of the five confounds from Section 2 is now active.**  
Your measurement from the second addition is not a true initial velocity.

---
### The correct architecture: one fresh flask per [S] value

```
Flask 1:  [S]0 =   5 mM  ->  measure v0  ->  one point on MM curve
Flask 2:  [S]0 =  10 mM  ->  measure v0  ->  one point on MM curve
Flask 3:  [S]0 =  20 mM  ->  measure v0  ->  one point on MM curve
Flask 4:  [S]0 =  40 mM  ->  measure v0  ->  one point on MM curve
Flask 5:  [S]0 =  70 mM  ->  measure v0  ->  one point on MM curve
Flask 6:  [S]0 = 110 mM  ->  measure v0  ->  one point on MM curve
Flask 7:  [S]0 = 160 mM  ->  measure v0  ->  one point on MM curve
Flask 8:  [S]0 = 220 mM  ->  measure v0  ->  one point on MM curve
                                               down arrow
                                     Plot all 8 points -> MM curve
```

Each flask:
- Contains **identical enzyme concentration**
- Has a **different starting [S]**
- Is measured **only at t ≈ 0** (the initial linear window)
- Is then **discarded** — the rest of the progress curve is not used

**The textbook's smooth curve required at minimum 8 separate experiments.**  

**Question before running the cell below:**  
*If you reuse the same flask (wrong way), will your measured rates be too high or too low? Why?*

In [ ]:
# === SECTION 3: Many Flasks — Right Way vs Wrong Way ===
from scipy.optimize import curve_fit

S0_values = [5, 10, 20, 40, 70, 110, 160, 220]
colors_flasks = plt.cm.viridis(np.linspace(0.15, 0.85, len(S0_values)))

# --- WRONG WAY: reuse one flask, add more substrate each time ---
# After each run, product has built up and enzyme has partially decayed
wrong_way_v0s = []
cumulative_P  = 0.0
cumulative_t  = 0.0
run_time_each = 8   # minutes between each substrate addition (too long)
Ki_p_wrong    = 40  # product inhibition constant

for s0 in S0_values:
    Keff  = Km * (1 + cumulative_P / Ki_p_wrong)  # Km elevated by product
    Veff  = Vmax * np.exp(-0.025 * cumulative_t)  # Vmax reduced by decay
    v_w   = (Veff * s0) / (Keff + s0)
    wrong_way_v0s.append(v_w)
    dP = v_w * run_time_each * 0.3
    cumulative_P = min(cumulative_P + dP, s0 * 0.8)
    cumulative_t += run_time_each

# --- RIGHT WAY: fresh flask for each [S] ---
right_way_v0s = [mm_rate(s, Vmax, Km) for s in S0_values]

# --- FIGURE ---
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(
    'Section 3 — The Architecture of the MM Experiment\n'
    'Why Every [S] Value Needs Its Own Fresh Flask',
    fontsize=13, fontweight='bold'
)

# --- Panel A: Right way — 8 progress curves ---
ax_prog = axes[0, 0]
for s0, col in zip(S0_values, colors_flasks):
    t, _, P, _ = simulate_progress_curve(s0, Vmax, Km, t_max=35)
    ax_prog.plot(t, P, color=col, linewidth=1.8, label=f'[S]={s0} mM')
    v0 = mm_rate(s0, Vmax, Km)
    t_tan = np.linspace(0, 1.8, 20)
    ax_prog.plot(t_tan, v0 * t_tan, color=col, linewidth=4.0, alpha=0.95)

ax_prog.axvspan(0, 1.8, alpha=0.12, color='#BA7517')
ax_prog.set_xlabel('Time (min)')
ax_prog.set_ylabel('[P] formed (mM)')
ax_prog.set_title(
    'RIGHT WAY: 8 separate fresh flasks\n'
    'Thick line = v0 extracted from initial slope only',
    fontsize=10, color='#085041'
)
ax_prog.legend(fontsize=7, ncol=2, title='Starting [S]')
ax_prog.text(2, 8, 'Only this\norange zone\nis used', fontsize=8,
              color='#BA7517', style='italic')

# --- Panel B: Wrong way — one flask, sequential additions ---
ax_wrong = axes[0, 1]
t_wf, _, P_wf, _ = simulate_progress_curve(
    S0_values[-1], Vmax, Km, t_max=35,
    product_inhibition=True, Ki_p=Ki_p_wrong,
    enzyme_decay=True, kd=0.025
)
t_ref2, _, P_ref2, _ = simulate_progress_curve(S0_values[-1], Vmax, Km, t_max=35)
ax_wrong.plot(t_wf,   P_wf,   color='#E24B4A', linewidth=2.2,
               label='Single reused flask')
ax_wrong.plot(t_ref2, P_ref2, color='#aaa',    linewidth=1.5,
               linestyle='--', label='Fresh flask reference')
for i, s0 in enumerate(S0_values):
    ax_wrong.axvline(i * run_time_each, color='#E24B4A', linestyle=':', alpha=0.4)
    ax_wrong.text(i * run_time_each + 0.3, 3, f'+{s0}mM',
                   fontsize=6.5, color='#E24B4A', rotation=90)
ax_wrong.set_xlabel('Time (min)')
ax_wrong.set_ylabel('[P] formed (mM)')
ax_wrong.set_title(
    'WRONG WAY: One flask, substrate added sequentially\n'
    'Each dotted line = next addition into corrupted mixture',
    fontsize=10, color='#791F1F'
)
ax_wrong.legend(fontsize=9)

# --- Panel C: Schematic — flask icons ---
ax_sch = axes[1, 0]
ax_sch.axis('off')
ax_sch.set_xlim(0, 10); ax_sch.set_ylim(0, 4)

ax_sch.text(5, 3.7, 'RIGHT WAY — each flask is completely fresh',
             ha='center', fontsize=10, fontweight='bold', color='#085041')
for i, (s0, col) in enumerate(zip(S0_values, colors_flasks)):
    x = 0.4 + i * 1.15
    rect = plt.Rectangle((x, 2.1), 0.85, 1.35, color=col, alpha=0.8,
                           ec='#333', lw=0.8)
    ax_sch.add_patch(rect)
    ax_sch.text(x + 0.42, 3.55, f'{s0}', ha='center', fontsize=7, color='#333')
    ax_sch.text(x + 0.42, 2.62, f'v0=\n{mm_rate(s0,Vmax,Km):.0f}',
                 ha='center', fontsize=6.5, color='white', fontweight='bold')
    ax_sch.text(x + 0.42, 2.02, 'FRESH', ha='center', fontsize=6, color='#085041')

ax_sch.text(5, 1.6, 'WRONG WAY — one flask, substrates added over time',
             ha='center', fontsize=10, fontweight='bold', color='#791F1F')
wrong_rect = plt.Rectangle((1.5, 0.2), 7, 1.2,
                             color='#FCEBEB', ec='#E24B4A', lw=1.5)
ax_sch.add_patch(wrong_rect)
for i, s0 in enumerate(S0_values):
    xw = 1.8 + i * 0.85
    ax_sch.annotate(f'+{s0}\nt={i*run_time_each}m',
                     xy=(xw, 0.8), ha='center', fontsize=6, color='#791F1F')
ax_sch.text(5, 0.05,
             'Product, decayed enzyme, reverse reaction — all accumulate',
             ha='center', fontsize=8, color='#E24B4A')

# --- Panel D: MM curves compared — right vs wrong ---
ax_cmp = axes[1, 1]
S_smooth = np.linspace(0.1, 240, 400)
v_true_curve = mm_rate(S_smooth, Vmax, Km)
ax_cmp.plot(S_smooth, v_true_curve, color='#1D9E75', linewidth=2.5,
             label=f'True MM curve  (Vmax={Vmax}, Km={Km})', zorder=4)
ax_cmp.scatter(S0_values, right_way_v0s, color='#1D9E75', s=90, zorder=5,
                label='v0 from fresh flasks (correct)')
ax_cmp.scatter(S0_values, wrong_way_v0s, color='#E24B4A', s=90,
                marker='x', linewidths=2.5, zorder=5,
                label='v0 from reused flask (wrong)')

for s, vr, vw in zip(S0_values, right_way_v0s, wrong_way_v0s):
    ax_cmp.annotate('', xy=(s, vw), xytext=(s, vr),
                     arrowprops=dict(arrowstyle='->', color='#E24B4A',
                                     lw=1.2, alpha=0.7))

try:
    popt_w, _ = curve_fit(mm_rate, S0_values, wrong_way_v0s,
                           p0=[Vmax, Km], maxfev=5000)
    v_wc = mm_rate(S_smooth, *popt_w)
    ax_cmp.plot(S_smooth, v_wc, color='#E24B4A', linewidth=2,
                 linestyle='--', alpha=0.7,
                 label=f'Fitted wrong curve (Vmax~{popt_w[0]:.0f}, Km~{popt_w[1]:.0f})')
    print(f'True parameters : Vmax = {Vmax},  Km = {Km}')
    print(f'Wrong-way fit   : Vmax ~ {popt_w[0]:.1f},  Km ~ {popt_w[1]:.1f}')
    print(f'Error in Vmax   : {abs(popt_w[0]-Vmax)/Vmax*100:.1f}%')
    print(f'Error in Km     : {abs(popt_w[1]-Km)/Km*100:.1f}%')
    print()
    print('If this were your soil enzyme data, you would report the WRONG Km.')
    print('That means you would draw incorrect conclusions about contamination effects.')
except Exception as e:
    print('Curve fit note:', e)

ax_cmp.axhline(Vmax, color='#BA7517', linestyle='--', alpha=0.4)
ax_cmp.set_xlabel('[S]0  (mM)')
ax_cmp.set_ylabel('v0  (umol/min)')
ax_cmp.set_title(
    'MM curve: right way vs wrong way\n'
    'Red arrows = error introduced by reusing flasks',
    fontsize=10
)
ax_cmp.legend(fontsize=7.5)

plt.tight_layout()
plt.savefig('section3_flask_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n ANSWER IN YOUR NOTEBOOK:')
print(f'  The MM experiment requires {len(S0_values)} separate flasks.')
print('  Each flask: identical enzyme amount, different [S]0, measured ONLY at t=0.')
print()
print('  1. Are the wrong-way rates too high or too low? Why in that direction?')
print('  2. Which confound caused the largest error in the wrong-way rates?')
print('  3. What Km would you report from wrong-way data?')
print('     What would that tell you (wrongly) about the enzyme health?')
print('  4. Design a simple check: how would you know if your flask was contaminated')
print('     by previous runs? (Hint: what could you add to test for product inhibition?)')


---
## Section 4 — The Data Demands a Shape
### Before writing any equation — what does the data look like, and what kind of curve fits it?

Forget the MM equation for now. You have data.  
**What mathematical properties must any curve have to fit this data?**

In [ ]:
# === SECTION 4: What shape does the data demand? ===

# Simulated 'experimental' data with realistic noise
np.random.seed(42)
S_exp = np.array([2, 5, 10, 20, 40, 80, 160, 300])
v_true = mm_rate(S_exp, Vmax, Km)
noise = np.random.normal(0, 2.5, len(S_exp))
v_obs = np.clip(v_true + noise, 0.5, Vmax + 5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Section 4 — What Shape Must the Equation Have?\n'
             'Compare three candidate curves against the data',
             fontsize=13, fontweight='bold')

S_fit = np.linspace(0, 320, 400)

# Candidate 1: Linear
ax1 = axes[0]
slope = np.polyfit(S_exp, v_obs, 1)
v_linear = np.polyval(slope, S_fit)
ax1.scatter(S_exp, v_obs, color='#1D9E75', s=70, zorder=5, label='Observed data')
ax1.plot(S_fit, v_linear, color='#E24B4A', linewidth=2, label='Linear fit')
ax1.axhline(Vmax, color='#BA7517', linestyle='--', alpha=0.4, label=f'Vmax = {Vmax}')
ax1.set_title('Candidate 1: Linear\nv = a·[S]', fontsize=11, color='#E24B4A')
ax1.set_xlabel('[S] (mM)'); ax1.set_ylabel('v₀ (µmol/min)')
ax1.set_ylim(-5, 100)
ax1.text(0.05, 0.92, 'WRONG\nNo ceiling — rate\ngrows forever',
          transform=ax1.transAxes, fontsize=9, color='#E24B4A',
          bbox=dict(boxstyle='round', facecolor='#FCEBEB', edgecolor='#E24B4A'))
ax1.legend(fontsize=9)

# Candidate 2: Quadratic
ax2 = axes[1]
coeffs2 = np.polyfit(S_exp, v_obs, 2)
v_quad = np.polyval(coeffs2, S_fit)
ax2.scatter(S_exp, v_obs, color='#1D9E75', s=70, zorder=5, label='Observed data')
ax2.plot(S_fit, v_quad, color='#BA7517', linewidth=2, label='Quadratic fit')
ax2.axhline(Vmax, color='#BA7517', linestyle='--', alpha=0.4, label=f'Vmax = {Vmax}')
ax2.set_title('Candidate 2: Quadratic\nv = a·[S]² + b·[S] + c', fontsize=11, color='#BA7517')
ax2.set_xlabel('[S] (mM)'); ax2.set_ylabel('v₀ (µmol/min)')
ax2.set_ylim(-5, 100)
ax2.text(0.05, 0.92, 'WRONG\nTurns back down\nat high [S]',
          transform=ax2.transAxes, fontsize=9, color='#BA7517',
          bbox=dict(boxstyle='round', facecolor='#FAEEDA', edgecolor='#BA7517'))
ax2.legend(fontsize=9)

# Candidate 3: Hyperbolic (MM)
ax3 = axes[2]
v_mm_fit = mm_rate(S_fit, Vmax, Km)
ax3.scatter(S_exp, v_obs, color='#1D9E75', s=70, zorder=5, label='Observed data')
ax3.plot(S_fit, v_mm_fit, color='#7F77DD', linewidth=2.5, label='Hyperbolic (MM) fit')
ax3.axhline(Vmax, color='#BA7517', linestyle='--', alpha=0.5, label=f'Vmax = {Vmax} (asymptote)')
ax3.axvline(Km, color='#1D9E75', linestyle=':', alpha=0.6, label=f'Km = {Km}')
ax3.set_title('Candidate 3: Hyperbolic\nv = Vmax·[S] / (Km + [S])', fontsize=11, color='#7F77DD')
ax3.set_xlabel('[S] (mM)'); ax3.set_ylabel('v₀ (µmol/min)')
ax3.set_ylim(-5, 100)
ax3.text(0.05, 0.92, 'CORRECT\nRises steeply,\napproaches ceiling',
          transform=ax3.transAxes, fontsize=9, color='#085041',
          bbox=dict(boxstyle='round', facecolor='#E1F5EE', edgecolor='#1D9E75'))
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig('section4_data_demands_shape.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔍 KEY INSIGHT — write this in your notebook:")
print("  The data has THREE properties any candidate equation must match:")
print("    (i)   v = 0 when [S] = 0  (no substrate, no reaction)")
print("    (ii)  v rises steeply at low [S]  (enzyme is not yet saturated)")
print("    (iii) v approaches a ceiling (Vmax) at high [S]  (enzyme is fully occupied)")
print()
print("  Linear fails property (iii). Quadratic fails (iii) badly.")
print("  Only the rectangular hyperbola satisfies all three.")
print("  The data cornered the equation — it had no other choice.")

---
## Section 5 — The Mechanism Explains the Shape
### Why does saturation exist? What is physically happening at the active site?

The hyperbolic shape is not a mathematical coincidence.  
It is a **portrait of saturation** — of a finite number of active sites filling up.

In [ ]:
# === SECTION 5: The Active Site Saturation Mechanism ===

n_sites = 10  # active sites per enzyme molecule
S_levels = [5, 15, 25, 60, 150, 400]   # mM
labels_S = ['Very low\n[S] = 5 mM',
             'Low\n[S] = 15 mM',
             'At Km\n[S] = 25 mM',
             'Moderate\n[S] = 60 mM',
             'High\n[S] = 150 mM',
             'Saturated\n[S] = 400 mM']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Section 5 — Saturation: Finite Active Sites Explain the Hyperbolic Curve',
             fontsize=13, fontweight='bold')

for idx, (S_val, lbl) in enumerate(zip(S_levels, labels_S)):
    ax = axes[idx // 3][idx % 3]
    fraction_occ = S_val / (Km + S_val)
    n_occupied = round(fraction_occ * n_sites)
    pct_vmax = fraction_occ * 100

    # Draw active site circles
    for site in range(n_sites):
        x = (site % 5) * 1.8 + 0.9
        y = 1.6 - (site // 5) * 1.6
        occupied = site < n_occupied
        circle = plt.Circle((x, y), 0.7,
                              color='#1D9E75' if occupied else '#e8e8e8',
                              ec='#085041' if occupied else '#aaaaaa',
                              linewidth=1.5, zorder=3)
        ax.add_patch(circle)
        ax.text(x, y, 'ES' if occupied else 'E',
                ha='center', va='center', fontsize=9,
                color='white' if occupied else '#888',
                fontweight='bold')

    # Rate bar
    bar_x = 9.8
    ax.barh(0.8, pct_vmax / 100 * 2, left=bar_x, height=1.4,
             color='#7F77DD', alpha=0.8)
    ax.barh(0.8, 2, left=bar_x, height=1.4,
             color='#e8e8e8', alpha=0.5)
    ax.text(bar_x + 1.0, 0.8, f'{pct_vmax:.0f}%\nVmax',
             ha='center', va='center', fontsize=9,
             color='white' if pct_vmax > 40 else '#534AB7', fontweight='bold')

    ax.set_xlim(0, 12.5)
    ax.set_ylim(-0.3, 2.8)
    ax.set_title(f'{lbl}\n{n_occupied}/{n_sites} sites occupied',
                  fontsize=10, color='#085041' if n_occupied >= n_sites//2 else '#888')
    ax.axis('off')

    occ_col = '#1D9E75' if n_occupied > 0 else '#aaa'
    ax.text(0.02, 0.02,
             f'v₀ = {mm_rate(S_val, Vmax, Km):.1f} µmol/min',
             transform=ax.transAxes, fontsize=9, color=occ_col)

plt.tight_layout()
plt.savefig('section5_mechanism.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔍 THE BIOLOGICAL MEANING — complete in your notebook:")
print("\n  Vmax is not just a parameter. It is the rate when:")
print("  ___________________________________________________")
print()
print("  Km is not just a parameter. It is the [S] at which:")
print("  ___________________________________________________")
print()
print(f"  At [S] = Km = {Km} mM, exactly {50}% of active sites are occupied.")
print(f"  Verify: mm_rate({Km}, {Vmax}, {Km}) = {mm_rate(Km, Vmax, Km):.1f}  ← should be {Vmax//2}")

---
## Section 6 — The Equation is Cornered
### Derive it from the mechanism. It cannot be anything else.

Now we derive the MM equation from first principles.  
The equation is not given to us — it is the **only algebraic expression consistent** with the mechanism and the data shape.

In [ ]:
# === SECTION 6: The Derivation Visualised ===

print("THE MICHAELIS-MENTEN DERIVATION — Step by Step")
print("=" * 55)
print()
print("STEP 1 — The mechanism:")
print("  E + S  ⇌  ES  →  E + P")
print("  k₁↓ k₋₁↑    kcat")
print()
print("STEP 2 — Rate is proportional to [ES]:")
print("  v = kcat × [ES]")
print()
print("STEP 3 — Steady state assumption ([ES] is roughly constant):")
print("  d[ES]/dt = 0")
print("  k₁[E][S] = (k₋₁ + kcat)[ES]")
print()
print("STEP 4 — Define Km = (k₋₁ + kcat) / k₁")
print("  [ES] = [E][S] / Km")
print()
print("STEP 5 — Conservation of enzyme: [E]total = [E] + [ES]")
print("  [E] = [E]total - [ES]")
print("  Substituting:")
print("  [ES] = [E]total × [S] / (Km + [S])")
print()
print("STEP 6 — Substitute into rate equation:")
print("  v = kcat × [E]total × [S] / (Km + [S])")
print("  Since Vmax = kcat × [E]total:")
print()
print("  ┌─────────────────────────────────────┐")
print("  │                                     │")
print("  │    v = Vmax × [S] / (Km + [S])      │")
print("  │                                     │")
print("  └─────────────────────────────────────┘")
print()
print("This is the ONLY equation consistent with:")
print("  ✓ The observed hyperbolic curve shape")
print("  ✓ The mechanism E + S ⇌ ES → E + P")
print("  ✓ The steady-state assumption")
print("  ✓ Conservation of total enzyme")

# Now verify numerically
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Section 6 — The Derived Equation Perfectly Fits the Cornered Shape',
             fontsize=13, fontweight='bold')

S_range = np.linspace(0, 300, 500)
v_derived = mm_rate(S_range, Vmax, Km)

ax1 = axes[0]
ax1.plot(S_range, v_derived, color='#7F77DD', linewidth=3, label='v = Vmax·[S]/(Km+[S])')
ax1.scatter(S_exp, v_obs, color='#1D9E75', s=80, zorder=5, label='Simulated experimental data')
ax1.axhline(Vmax, color='#BA7517', linestyle='--', alpha=0.5, linewidth=1.5,
            label=f'Vmax = {Vmax} µmol/min')
ax1.axhline(Vmax/2, color='#1D9E75', linestyle=':', alpha=0.5, linewidth=1.5)
ax1.axvline(Km, color='#1D9E75', linestyle=':', alpha=0.5, linewidth=1.5,
            label=f'Km = {Km} mM')
ax1.annotate(f'At [S]=Km={Km}, v=Vmax/2={Vmax//2}',
              xy=(Km, Vmax/2), xytext=(Km+40, Vmax/2-12),
              arrowprops=dict(arrowstyle='->', color='#085041'), fontsize=9, color='#085041')
ax1.set_xlabel('[S]  (mM)')
ax1.set_ylabel('v₀  (µmol/min)')
ax1.set_title('The derived equation fits the data perfectly', fontsize=11)
ax1.legend(fontsize=9)

# Lineweaver-Burk — the linearisation
ax2 = axes[1]
S_lb = np.array([5, 8, 12, 20, 35, 60, 100, 180])
v_lb = mm_rate(S_lb, Vmax, Km) + np.random.normal(0, 0.5, len(S_lb))
inv_S = 1 / S_lb
inv_v = 1 / v_lb

# Fit line
coeffs = np.polyfit(inv_S, inv_v, 1)
x_line = np.linspace(-0.06, 0.22, 200)
y_line = np.polyval(coeffs, x_line)

ax2.scatter(inv_S, inv_v, color='#7F77DD', s=70, zorder=5, label='1/v vs 1/[S]')
ax2.plot(x_line, y_line, color='#D85A30', linewidth=2, label='Linear fit')
ax2.axvline(0, color='#aaa', linewidth=0.8)
ax2.axhline(0, color='#aaa', linewidth=0.8)

x_int = -coeffs[1]/coeffs[0]
y_int = coeffs[1]
ax2.scatter([x_int], [0], color='#E24B4A', s=100, zorder=6)
ax2.scatter([0], [y_int], color='#1D9E75', s=100, zorder=6)
ax2.annotate(f'x-int = −1/Km\n= {x_int:.4f}  →  Km ≈ {-1/x_int:.0f} mM',
              xy=(x_int, 0), xytext=(x_int+0.04, 0.006),
              arrowprops=dict(arrowstyle='->', color='#E24B4A'), fontsize=9, color='#E24B4A')
ax2.annotate(f'y-int = 1/Vmax\n= {y_int:.4f}  →  Vmax ≈ {1/y_int:.0f}',
              xy=(0, y_int), xytext=(0.04, y_int+0.003),
              arrowprops=dict(arrowstyle='->', color='#1D9E75'), fontsize=9, color='#1D9E75')
ax2.set_xlabel('1/[S]  (mM⁻¹)')
ax2.set_ylabel('1/v  (min/µmol)')
ax2.set_title('Lineweaver-Burk: taking reciprocals\nstraightens the curve → easy parameter reading', fontsize=11)
ax2.legend(fontsize=9)
ax2.set_ylim(-0.01, 0.055)

plt.tight_layout()
plt.savefig('section6_derivation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — The Equation as a Tool: Reading Biology from Numbers
### Km and Vmax are not abstract constants — they carry biological meaning

In [ ]:
# === SECTION 7: Biological Meaning of Parameters ===
# Compare real enzyme scenarios

enzymes = [
    dict(name='Healthy soil urease\n(Odisha agricultural soil)',
         Vmax=90, Km=18, color='#1D9E75',
         note='Low Km — efficient at low urea.\nHealthy soil functions well.'),
    dict(name='Urease from\nindustrially contaminated soil',
         Vmax=55, Km=62, color='#E24B4A',
         note='High Km + lower Vmax.\nHeavy metals disrupting enzyme.'),
    dict(name='Earthworm gut urease\n(Drawida sp., Western Odisha)',
         Vmax=110, Km=28, color='#7F77DD',
         note='Higher Vmax — gut environment\noptimised for nitrogen cycling.'),
    dict(name='Alkaline phosphatase\n(post-monsoon, paddy soil)',
         Vmax=70, Km=12, color='#BA7517',
         note='Very low Km — high affinity\nfor phosphate in seasonal soils.'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Section 7 — Parameters Carry Biological Meaning\n'
             'Reading enzyme health from Km and Vmax',
             fontsize=13, fontweight='bold')

S_range = np.linspace(0, 250, 400)
ax1, ax2 = axes

for e in enzymes:
    v = mm_rate(S_range, e['Vmax'], e['Km'])
    ax1.plot(S_range, v, color=e['color'], linewidth=2.2,
              label=e['name'].replace('\n', ' '))
    ax1.axvline(e['Km'], color=e['color'], linestyle=':', alpha=0.35, linewidth=1)

ax1.set_xlabel('[Substrate]  (mM)')
ax1.set_ylabel('v₀  (µmol/min)')
ax1.set_title('Same equation — different biology\nEach curve tells a different ecological story', fontsize=11)
ax1.legend(fontsize=8, loc='lower right')

# Parameter comparison bar chart
names = [e['name'].split('\n')[0] for e in enzymes]
Kms   = [e['Km']   for e in enzymes]
Vmaxs = [e['Vmax'] for e in enzymes]
colors = [e['color'] for e in enzymes]
x = np.arange(len(enzymes))
width = 0.35

bars1 = ax2.bar(x - width/2, Kms, width, label='Km (mM)', color=colors, alpha=0.7)
bars2 = ax2.bar(x + width/2, Vmaxs, width, label='Vmax (µmol/min)',
                 color=colors, alpha=0.4, hatch='//')
ax2.set_xticks(x)
ax2.set_xticklabels([n[:22] for n in names], fontsize=8, rotation=15, ha='right')
ax2.set_ylabel('Value')
ax2.set_title('Km and Vmax as ecological indicators\n(lower Km = higher substrate affinity)', fontsize=11)
ax2.legend(fontsize=9)

for bar, e in zip(bars1, enzymes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
              f'{e["Km"]}', ha='center', va='bottom', fontsize=8, color=e['color'])
for bar, e in zip(bars2, enzymes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
              f'{e["Vmax"]}', ha='center', va='bottom', fontsize=8, color=e['color'])

plt.tight_layout()
plt.savefig('section7_biological_meaning.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🌍 OSHEC RESEARCH CONNECTION:")
print("  In your soil biomarker project, if you measure soil enzyme Km values:")
print("  • Elevated Km in contaminated soil = heavy metal disrupting active site affinity")
print("  • Reduced Vmax = fewer active enzyme molecules (denaturation / inhibition)")
print("  • The MM equation is not just a textbook formula —")
print("    it is the diagnostic tool for reading enzyme health from field data.")
for e in enzymes:
    eff = e['Vmax'] / e['Km']
    print(f"  Catalytic efficiency ({e['name'].split(chr(10))[0]}): Vmax/Km = {eff:.1f}")

---
## Section 8 — 🧪 Your Own Experiment
### Change the parameters. Hunt the pattern yourself.

**Instructions:** Modify the values in the cell below and re-run.  
Predict what will happen *before* you run. Then check if you were right.

In [ ]:
# ============================================================
# 🧪 STUDENT EXPERIMENT — MODIFY THESE VALUES AND RE-RUN
# ============================================================

# --- Your enzyme parameters ---
MY_Vmax = 80     # Try: 40, 80, 120  — what changes?
MY_Km   = 25     # Try: 5, 25, 80   — what changes?
MY_S0   = 100    # Starting substrate for progress curve

# --- Confounding factors (set True to turn on) ---
PRODUCT_INHIBITION = False   # Ki_p below controls its strength
Ki_p               = 40      # mM — lower = stronger inhibition
ENZYME_DECAY       = False   # kd below controls decay speed
kd                 = 0.025   # min⁻¹
REVERSE_REACTION   = False   # Keq below
Keq                = 5       # equilibrium constant

# ============================================================
# WRITE YOUR PREDICTION HERE BEFORE RUNNING:
# "I expect the progress curve to ____________"
# "I expect the MM curve to ____________"
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'Your Experiment — Vmax={MY_Vmax}, Km={MY_Km}, [S]₀={MY_S0} mM',
             fontsize=13, fontweight='bold')

# Progress curve
t, _, P, v = simulate_progress_curve(
    MY_S0, MY_Vmax, MY_Km,
    product_inhibition=PRODUCT_INHIBITION, Ki_p=Ki_p,
    enzyme_decay=ENZYME_DECAY, kd=kd,
    reverse_reaction=REVERSE_REACTION, Keq=Keq
)
t_ideal, _, P_ideal, _ = simulate_progress_curve(MY_S0, MY_Vmax, MY_Km)

ax1 = axes[0]
ax1.plot(t, P_ideal, color='#aaa', linestyle='--', linewidth=1.5, label='Ideal (no confounds)')
ax1.plot(t, P, color='#1D9E75', linewidth=2.5, label='Your experiment')
v0_my = mm_rate(MY_S0, MY_Vmax, MY_Km)
t_w = np.linspace(0, 2, 30)
ax1.plot(t_w, v0_my * t_w, color='#BA7517', linewidth=3, label=f'v₀ = {v0_my:.1f}')
ax1.set_xlabel('Time (min)'); ax1.set_ylabel('[P] (mM)')
ax1.set_title('Progress curve', fontsize=11)
ax1.legend(fontsize=9)

# MM curve
ax2 = axes[1]
S_r = np.linspace(0, MY_Km * 10, 400)
v_r = mm_rate(S_r, MY_Vmax, MY_Km)
ax2.plot(S_r, v_r, color='#7F77DD', linewidth=2.5)
ax2.axhline(MY_Vmax, color='#BA7517', linestyle='--', alpha=0.5, label=f'Vmax={MY_Vmax}')
ax2.axhline(MY_Vmax/2, color='#1D9E75', linestyle=':', alpha=0.5)
ax2.axvline(MY_Km, color='#1D9E75', linestyle=':', alpha=0.5, label=f'Km={MY_Km}')
ax2.set_xlabel('[S] (mM)'); ax2.set_ylabel('v₀ (µmol/min)')
ax2.set_title('MM curve from your parameters', fontsize=11)
ax2.legend(fontsize=9)

# Lineweaver-Burk
ax3 = axes[2]
S_lb = np.linspace(MY_Km * 0.2, MY_Km * 10, 10)
v_lb = mm_rate(S_lb, MY_Vmax, MY_Km)
ax3.scatter(1/S_lb, 1/v_lb, color='#D85A30', s=60, zorder=5, label='Data points')
inv_x = np.linspace(-1/(MY_Km*0.8), 1/(MY_Km*0.2), 200)
inv_y = (MY_Km / MY_Vmax) * inv_x + (1 / MY_Vmax)
ax3.plot(inv_x, inv_y, color='#D85A30', linewidth=2, label='Fit')
ax3.axvline(0, color='#aaa', linewidth=0.8)
ax3.axhline(0, color='#aaa', linewidth=0.8)
ax3.scatter([-1/MY_Km], [0], color='#E24B4A', s=100, zorder=6,
             label=f'x-int = −1/Km = {-1/MY_Km:.4f}')
ax3.scatter([0], [1/MY_Vmax], color='#1D9E75', s=100, zorder=6,
             label=f'y-int = 1/Vmax = {1/MY_Vmax:.4f}')
ax3.set_xlabel('1/[S]'); ax3.set_ylabel('1/v')
ax3.set_title('Lineweaver-Burk plot\n(verify your parameters)', fontsize=11)
ax3.legend(fontsize=7.5)

plt.tight_layout()
plt.show()

print("\n📋 YOUR RESULTS SUMMARY:")
print(f"  Vmax set           = {MY_Vmax} µmol/min")
print(f"  Km set             = {MY_Km} mM")
print(f"  v₀ at [S]₀={MY_S0}   = {v0_my:.1f} µmol/min")
print(f"  Catalytic eff.     = {MY_Vmax/MY_Km:.2f}  (Vmax/Km)")
print(f"  Confounds active   : product_inhibition={PRODUCT_INHIBITION}, enzyme_decay={ENZYME_DECAY}, reverse={REVERSE_REACTION}")
print()
print("📝 REFLECTION QUESTIONS:")
print("  1. How did turning on product inhibition change the progress curve shape?")
print("  2. Which parameter (Km or Vmax) most affects efficiency at low [S]?")
print("  3. Design an experiment to distinguish product inhibition from enzyme decay.")
print("     (Hint: what would change if you added fresh enzyme partway through?)")

---
## Section 9 — The Complete Picture the Textbook Never Shows
### All curve types side by side — with the progress curve that generates each one

In [ ]:
# === SECTION 9: The Figure Textbooks Should Show ===

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)
fig.suptitle('The Complete Picture — What One Textbook Figure Hides\n'
             'Top row: progress curves (the actual experiment) | '
             'Bottom row: derived v₀ curves (what textbooks show)',
             fontsize=12, fontweight='bold')

curve_types = [
    dict(label='Simple MM\n(hyperbolic)',
         color='#1D9E75',
         badge='Textbooks show this one',
         v_fn=lambda S: mm_rate(S, 80, 25),
         prog_kwargs=dict(S0=120, Vmax=80, Km=25),
         note='Most enzymes:\nurease, amylase, lysozyme'),
    dict(label='Cooperative\n(sigmoidal)',
         color='#7F77DD',
         badge='Missing from most textbooks',
         v_fn=lambda S: 80 * S**2.8 / (25**2.8 + S**2.8),
         prog_kwargs=dict(S0=120, Vmax=80, Km=25),
         note='Allosteric enzymes:\nPFK, ATCase, hemoglobin'),
    dict(label='Substrate inhibition\n(bell-shaped)',
         color='#BA7517',
         badge='Rarely shown',
         v_fn=lambda S: mm_rate(S, 80, 25) / (1 + S/60) if S > 0 else 0,
         prog_kwargs=dict(S0=120, Vmax=80, Km=25,
                          product_inhibition=True, Ki_p=25),
         note='AChE, invertase,\ntyrosinase'),
    dict(label='Bisubstrate\n(two Km values)',
         color='#E24B4A',
         badge='Almost never shown',
         v_fn=lambda S: mm_rate(S, 35, 5) + mm_rate(S, 55, 65),
         prog_kwargs=dict(S0=120, Vmax=80, Km=25),
         note='Soil phosphatase,\nhexokinase, LDH'),
]

S_range = np.linspace(0, 250, 400)
t_range = np.linspace(0, 50, 300)

for col_idx, ct in enumerate(curve_types):
    # Progress curve (top row)
    ax_top = fig.add_subplot(gs[0, col_idx])
    kw = ct['prog_kwargs']
    t, _, P, _ = simulate_progress_curve(**kw)
    ax_top.plot(t, P, color=ct['color'], linewidth=2)
    ax_top.fill_between(t, P, alpha=0.08, color=ct['color'])
    ax_top.axvspan(0, 2, alpha=0.2, color='#BA7517')
    ax_top.set_xlabel('Time (min)', fontsize=9)
    ax_top.set_ylabel('[P] (mM)', fontsize=9)
    ax_top.set_title(f'{ct["label"]}\nProgress curve', fontsize=9,
                      color=ct['color'], fontweight='bold')
    ax_top.text(0.5, 0.07, 'v₀ here', transform=ax_top.transAxes,
                 fontsize=7, color='#BA7517', ha='center')

    # v₀ curve (bottom row)
    ax_bot = fig.add_subplot(gs[1, col_idx])
    v_vals = np.array([ct['v_fn'](s) for s in S_range])
    v_mm   = mm_rate(S_range, 80, 25)

    if col_idx > 0:
        ax_bot.plot(S_range, v_mm, color='#aaaaaa', linewidth=1.2,
                     linestyle='--', alpha=0.7, label='MM reference')
    ax_bot.plot(S_range, v_vals, color=ct['color'], linewidth=2.5,
                 label=ct['label'].replace('\n',' '))
    ax_bot.fill_between(S_range, v_vals, alpha=0.07, color=ct['color'])
    ax_bot.set_xlabel('[S] (mM)', fontsize=9)
    ax_bot.set_ylabel('v₀ (µmol/min)', fontsize=9)
    ax_bot.set_title(f'v₀ vs [S] curve', fontsize=9, color=ct['color'])
    ax_bot.text(0.97, 0.08, ct['note'], transform=ax_bot.transAxes,
                 fontsize=7.5, ha='right', va='bottom', color=ct['color'],
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                            edgecolor=ct['color'], alpha=0.8))
    ax_bot.text(0.03, 0.92, ct['badge'], transform=ax_bot.transAxes,
                 fontsize=7, ha='left', va='top',
                 color='#085041' if col_idx==0 else '#791F1F',
                 bbox=dict(boxstyle='round,pad=0.3',
                            facecolor='#E1F5EE' if col_idx==0 else '#FCEBEB',
                            edgecolor='none', alpha=0.9))
    if col_idx > 0:
        ax_bot.legend(fontsize=7, loc='center right')

plt.savefig('section9_complete_picture.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔑 FINAL INSIGHT:")
print("  The textbook's single figure is not wrong — it is incomplete.")
print("  It shows ONE of FOUR common curve shapes.")
print("  More importantly, it shows the DERIVED figure without the RAW experiment")
print("  (the progress curve) that actually produces it.")
print()
print("  A pattern hunter always asks:")
print("  'What experiment produced this figure?'")
print("  'What assumptions were made to get from raw data to this curve?'")
print("  'What other shapes might data from a different enzyme produce?'")

---
## 🌍 Section 10 — Western Odisha Connection
### Soil enzyme kinetics as environmental biomarkers

Everything in this notebook is directly relevant to monitoring soil health  
in the mining and industrial zones of Western Odisha.

**Your field question:** *How do Km and Vmax of soil enzymes change as a function of heavy metal contamination?*

In [ ]:
# === SECTION 10: Soil Enzyme Biomarkers — Western Odisha ===

# Hypothetical data based on published contamination gradients
# (students can replace with real field data)

contamination_levels = {
    'Control\n(forest soil)':       dict(Vmax=95,  Km=15,  color='#1D9E75'),
    'Low contamination\n(5 km)':    dict(Vmax=78,  Km=24,  color='#639922'),
    'Moderate\n(2 km from plant)':  dict(Vmax=58,  Km=42,  color='#BA7517'),
    'High\n(0.5 km from plant)':    dict(Vmax=38,  Km=71,  color='#D85A30'),
    'Severely contaminated\n(site)':dict(Vmax=18,  Km=110, color='#E24B4A'),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Section 10 — Soil Urease Kinetics as Contamination Biomarker\n'
             'Western Odisha Industrial Zone (hypothetical gradient data)',
             fontsize=12, fontweight='bold')

S_range = np.linspace(0, 250, 400)
ax1, ax2, ax3 = axes

for site, params in contamination_levels.items():
    v = mm_rate(S_range, params['Vmax'], params['Km'])
    ax1.plot(S_range, v, color=params['color'], linewidth=2.2,
              label=site.replace('\n',' '))

ax1.set_xlabel('[Urea]  (mM)')
ax1.set_ylabel('Urease activity  (µmol urea/min/g soil)')
ax1.set_title('Urease kinetics along\ncontamination gradient', fontsize=11)
ax1.legend(fontsize=7.5, loc='lower right')

# Km as biomarker
sites_short = ['Control', 'Low', 'Moderate', 'High', 'Severe']
Kms   = [p['Km']   for p in contamination_levels.values()]
Vmaxs = [p['Vmax'] for p in contamination_levels.values()]
colors_bar = [p['color'] for p in contamination_levels.values()]

bars = ax2.bar(sites_short, Kms, color=colors_bar, alpha=0.85, edgecolor='white')
ax2.set_ylabel('Km  (mM)  — lower = healthier', fontsize=10)
ax2.set_title('Km rises with contamination\n(enzyme loses substrate affinity)', fontsize=11)
for bar, val in zip(bars, Kms):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
              str(val), ha='center', fontsize=9, fontweight='bold')
ax2.tick_params(axis='x', rotation=20)

# Catalytic efficiency as integrated biomarker
efficiencies = [p['Vmax']/p['Km'] for p in contamination_levels.values()]
bars3 = ax3.bar(sites_short, efficiencies, color=colors_bar, alpha=0.85, edgecolor='white')
ax3.set_ylabel('Vmax/Km  (catalytic efficiency)', fontsize=10)
ax3.set_title('Catalytic efficiency as single\nintegrated soil health index', fontsize=11)
for bar, val in zip(bars3, efficiencies):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
              f'{val:.1f}', ha='center', fontsize=9, fontweight='bold')
ax3.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('section10_biomarker.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🌍 RESEARCH DESIGN EXERCISE:")
print("  You are Susama Kar, PhD scholar, measuring soil enzyme kinetics")
print("  at earthworm collection sites near Jharsuguda industrial zone.")
print()
print("  You collect soil from 5 sites with increasing distance from the plant.")
print("  For each soil, you run 8 MM experiments (different [urea] values).")
print()
print("  1. How many total progress curves will you record? ________")
print("  2. What is the minimum time window for valid v₀ measurement? ________")
print("  3. Which parameter — Km or Vmax — do you expect to change more with")
print("     heavy metal contamination, and why? ________")
print("  4. How would product inhibition by ammonia affect your urease assay?")
print("     How would you control for it? ________")
print()
print("  Replace the hypothetical data in this cell with your real field data")
print("  and this notebook becomes a live analysis tool for your OSHEC project.")

---
## Summary — What You Have Hunted

| Stage | What you did | What emerged |
|-------|-------------|-------------|
| 1 | Watched product form in one flask | The progress curve — the raw truth |
| 2 | Asked why late measurements fail | Five simultaneous confounds |
| 3 | Asked how MM curve is built | Many separate flasks, each giving one point |
| 4 | Let the data reveal its shape | Hyperbolic — must have a ceiling |
| 5 | Asked why the ceiling exists | Active site saturation — a finite resource |
| 6 | Derived from the mechanism | The MM equation — no other form is possible |
| 7 | Read parameters as biology | Km = affinity, Vmax = capacity, Vmax/Km = efficiency |
| 8 | Ran your own experiment | Parameters have consequences you can predict |
| 9 | Saw all curve shapes together | The textbook's one curve hides three others |
| 10 | Connected to real soils | The equation is a field diagnostic tool |

---

> **The Michaelis-Menten equation was not invented.**  
> It was cornered — backed into existence by observations that had no other honest explanation.
> 
> That is how all mathematical biology works.  
> And that is what it means to be a Pattern Hunter.

---
*Notebook developed for Pattern Hunters pedagogy | Kuchinda Degree College, Dept. of Zoology*  
*OSHEC Research Project: Genomic Responses of Earthworms to Industrial Pollution in Western Odisha*